# Module 07: Aircraft Software Control
**AE2.021 · USTH**

This notebook rewrites the 8-bit checksum and CRC-8 in code, reproduces Floyd Example 2-41, tries three error types (bit flip, swap, cancelling errors), simulates the FLS/DFLD release decision from Tooley 13.3.3, checks the level A to E numbers and cross-checks Tooley's printed chapter 13 answer key (read directly from the scanned Appendix 3 image, A.13, page 380).

## 1. 8-bit checksum and CRC-8

In [1]:
def checksum8(b): return sum(b) & 0xFF
def crc8(b, poly=0x07):
    c = 0
    for x in b:
        c ^= x
        for _ in range(8):
            c = ((c << 1) ^ poly) & 0xFF if c & 0x80 else (c << 1) & 0xFF
    return c

# Standard CRC-8 value (poly 0x07) of the ASCII string '123456789' is 0xF4
assert crc8(b"123456789") == 0xF4
data = bytes.fromhex("48 65 6C 6C 6F 20 45 45 43 21")      # same file as Demo 1 on the page
print("checksum8 =", hex(checksum8(data)), " crc8 =", hex(crc8(data)))
assert checksum8(bytes([10, 20, 30, 40, 50])) == 150        # generated question of the module

checksum8 = 0x2  crc8 = 0x36


## 2. Three error types

In [2]:
def verdict(orig, bad):
    return dict(checksum_sees=checksum8(bad) != checksum8(orig), crc_sees=crc8(bad) != crc8(orig))

flip = bytearray(data); flip[5] ^= 0x04
swap = bytearray(data); swap[1], swap[8] = swap[8], swap[1]
comp = bytearray(data); comp[0] = (comp[0] + 1) & 255; comp[9] = (comp[9] - 1) & 255
for name, bad in (("flip 1 bit", flip), ("swap 2 bytes", swap), ("two cancelling errors", comp)):
    print(f"{name:20s}", verdict(data, bad))
assert verdict(data, flip) == dict(checksum_sees=True, crc_sees=True)
assert verdict(data, swap) == dict(checksum_sees=False, crc_sees=True)     # the checksum is 'blind' to swaps
assert verdict(data, comp) == dict(checksum_sees=False, crc_sees=True)

# Extension: does CRC-8 catch EVERY 1-bit error and every swap of 2 different bytes? (exhaustive test)
import itertools
missed_flip = sum(1 for i in range(len(data)) for k in range(8)
                  if crc8(bytes(x ^ (1 << k) if j == i else x for j, x in enumerate(data))) == crc8(data))
print("1-bit errors missed by CRC:", missed_flip, "out of", len(data) * 8)
assert missed_flip == 0
miss_swap = [(i, j) for i, j in itertools.combinations(range(len(data)), 2)
             if data[i] != data[j] and crc8(bytes(data[j] if t == i else data[i] if t == j else x for t, x in enumerate(data))) == crc8(data)]
print("swapped pairs (different bytes) missed by CRC-8:", miss_swap, "- the checksum misses ALL of those pairs")

flip 1 bit           {'checksum_sees': True, 'crc_sees': True}
swap 2 bytes         {'checksum_sees': False, 'crc_sees': True}
two cancelling errors {'checksum_sees': False, 'crc_sees': True}
1-bit errors missed by CRC: 0 out of 80
swapped pairs (different bytes) missed by CRC-8: [] - the checksum misses ALL of those pairs


## 3. Floyd Example 2-41

In [3]:
def mod2_remainder(bits, g):
    b = [int(c) for c in bits]; gg = [int(c) for c in g]
    for i in range(len(b) - len(gg) + 1):
        if b[i]:
            for j, x in enumerate(gg): b[i + j] ^= x
    return "".join(map(str, b[-(len(gg) - 1):]))
D, G = "11010011", "1010"
appended = D + "0" * len(G)                         # Floyd appends 4 zeros (the number of bits of G)
rem = mod2_remainder(appended, G)
print("remainder:", rem)                                    # '100' with the last 3 bits; Floyd writes 0100 (4 bits)
assert rem == "100" and rem.zfill(4) == "0100"
tx = D + rem.zfill(4)
print("transmitted code:", tx)
assert tx == "110100110100"
assert mod2_remainder(tx, G) == "000"               # the receiver divides again and gets remainder 0
# The standard CRC version appends only len(G)-1 = 3 zeros:
std_rem = mod2_remainder(D + "000", G); print("Standard CRC (3 zeros):", std_rem, "-> transmitted code", D + std_rem)
assert mod2_remainder(D + std_rem, G) == "000"
# Parity (Floyd Table 2-8): the even-parity bit for BCD 0111 is 1
assert bin(0b0111).count("1") % 2 == 1

remainder: 100
transmitted code: 110100110100
Standard CRC (3 zeros): 010 -> transmitted code 11010011010


## 4. FLS/DFLD release decision (Tooley 13.3.3)

In [4]:
def release_document(kind, airworthiness_requirement=True, nav_database=False):
    """Return the release document accompanying the item per Tooley 13.3.3."""
    if kind not in ("FLS", "LSAP", "DFLD"): raise ValueError(kind)
    if nav_database: return "Letter of Acceptance (Form 1 or 8130-3 cannot be issued)"
    if kind == "LSAP" or (kind == "DFLD" and airworthiness_requirement): return "EASA Form 1 or FAA 8130-3"
    return "Certificate of Conformity"
assert release_document("LSAP") == "EASA Form 1 or FAA 8130-3"                       # EEC, DFDAU, ECU APU, FGC
assert release_document("DFLD", True) == "EASA Form 1 or FAA 8130-3"                 # FMC, TAWS, IMA
assert release_document("DFLD", True, nav_database=True).startswith("Letter of Acceptance")
assert release_document("FLS", False) == "Certificate of Conformity"

def may_load(suspect_corrupt, version_ok):
    return "QUARANTINE (do not transfer)" if suspect_corrupt or not version_ok else "can be loaded"
assert may_load(True, True).startswith("QUARANTINE") and may_load(False, True) == "can be loaded"
print("Release and quarantine rules: OK")

Release and quarantine rules: OK


## 5. Levels A to E

In [5]:
LEVELS = {"A": (None, 1e-9), "B": (1e-9, 1e-7), "C": (1e-7, 1e-5), "D": (1e-5, None)}   # (min, max) probability per flight hour, Tooley table 13.1
def level_for(p):
    for k, (lo, hi) in LEVELS.items():
        if (lo is None or p >= lo) and (hi is None or p < hi): return k
assert level_for(1e-10) == "A" and level_for(5e-9) == "B" and level_for(1e-6) == "C" and level_for(1e-3) == "D"
# Mandatory number of traces per figure 13-1: red (A) 2, purple (A,B,C) 3, green (A-D) 3 group arrows
TRACES = {"red": ("A",), "purple": ("A", "B", "C"), "green": ("A", "B", "C", "D")}
need = {lv: [c for c, lvls in TRACES.items() if lv in lvls] for lv in "ABCDE"}
print(need)
assert need["A"] == ["red", "purple", "green"] and need["B"] == need["C"] == ["purple", "green"] and need["D"] == ["green"] and need["E"] == []

{'A': ['red', 'purple', 'green'], 'B': ['purple', 'green'], 'C': ['purple', 'green'], 'D': ['green'], 'E': []}


## 6. Tooley Ch.13 answer key

In [6]:
KEY = dict(zip(range(1, 8), "c b c c b a b".split()))      # A.13 trang 380
# Questions that can be self-checked: 
assert KEY[2] == "b"      # Level B = extremely remote (table 13.1)
assert KEY[4] == "c"      # radar thoi tiet = Class D (table 13.2), la lua chon (c)
assert KEY[5] == "b"      # EEC = LSAP
assert KEY[6] == "a"      # OSS and UMS are classes of FLS
assert KEY[7] == "b"      # the final step of loading the EEC is verification
print("5/5 self-checkable questions match; questions 1 and 3 are definition questions (c) and (c) per the text of Tooley 13.1-13.2.")

5/5 self-checkable questions match; questions 1 and 3 are definition questions (c) and (c) per the text of Tooley 13.1-13.2.


#### 📤 Takeaway
All computed numbers agree. The checksum misses every swap and cancelling errors; CRC-8 caught all three error types tested, but a CRC can also miss other errors (not absolute). The lecture's printed answer to question 13-6 omits the second red arrow of Figure 13-1.